# Phase 1 - Data Exploration (KITTI 3D Object Detection)

Explore the LiDAR point clouds and labels before training:
class distribution, object distances and sizes, occlusion, point density,
and a bird's-eye-view of one scan with ground-truth boxes.

Every figure is saved to `results/` so it can go straight onto the slides.

**Velodyne coordinate frame:** `x` = forward, `y` = left, `z` = up.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# VS Code odpala kernel z katalogu domowego, więc nie ufamy cwd.
# Kotwiczymy się na lokalizacji notebooka (__vsc_ipynb_file__),
# a w razie czego próbujemy katalogu roboczego.
search_starts = []
if "__vsc_ipynb_file__" in globals():
    search_starts.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
search_starts.append(Path.cwd().resolve())

ROOT = None
for start in search_starts:
    p = start
    while True:
        if (p / "src" / "kitti_utils.py").exists():
            ROOT = p
            break
        if p == p.parent:
            break
        p = p.parent
    if ROOT is not None:
        break

if ROOT is None:
    # Ostatnia deska ratunku — ustaw ręcznie ścieżkę do projektu:
    ROOT = Path.home() / "Documents" / "ML" / "lidar-3d-detection"

assert (ROOT / "src" / "kitti_utils.py").exists(), f"Zły katalog główny: {ROOT}"

sys.path.insert(0, str(ROOT / "src"))
from kitti_utils import load_point_cloud, load_labels, load_calib, collect_all_labels

DATA = ROOT / "data" / "kitti"
TRAIN = DATA / "training"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
print("Project root:", ROOT)
print("Training data:", TRAIN)

FileNotFoundError: Nie znaleziono katalogu głównego (src/kitti_utils.py).

## 1. A single point cloud

In [ ]:
sample_id = "000010"
pc = load_point_cloud(TRAIN / "velodyne" / f"{sample_id}.bin")

print(f"Points in scan {sample_id}: {pc.shape[0]:,}")
print(f"x (forward): [{pc[:,0].min():6.1f}, {pc[:,0].max():6.1f}] m")
print(f"y (left):    [{pc[:,1].min():6.1f}, {pc[:,1].max():6.1f}] m")
print(f"z (up):      [{pc[:,2].min():6.1f}, {pc[:,2].max():6.1f}] m")
print(f"intensity:   [{pc[:,3].min():6.2f}, {pc[:,3].max():6.2f}]")

## 2. Class distribution (the headline)

`DontCare` is not a real object class - it marks regions left unannotated and
is ignored during evaluation. The three evaluated classes are
**Car, Pedestrian, Cyclist**. Note the strong imbalance.

In [ ]:
labels = collect_all_labels(TRAIN / "label_2")
counts = labels["type"].value_counts()
print(counts)

fig, ax = plt.subplots(figsize=(8, 4))
counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Object count per class (KITTI training)")
ax.set_xlabel("class")
ax.set_ylabel("count")
plt.tight_layout()
fig.savefig(RESULTS / "class_distribution.png", dpi=150)
plt.show()

In [ ]:
EVAL_CLASSES = ["Car", "Pedestrian", "Cyclist"]
eval_labels = labels[labels["type"].isin(EVAL_CLASSES)].copy()
print(eval_labels["type"].value_counts())

## 3. How far away are the objects?

In [ ]:
# Ground-plane distance from the sensor, in camera coords: sqrt(x^2 + z^2)
eval_labels["distance"] = np.sqrt(eval_labels["loc_x"]**2 + eval_labels["loc_z"]**2)

fig, ax = plt.subplots(figsize=(8, 4))
for cls in EVAL_CLASSES:
    sub = eval_labels[eval_labels["type"] == cls]
    ax.hist(sub["distance"], bins=40, alpha=0.5, label=cls)
ax.set_title("Object distance from sensor")
ax.set_xlabel("distance [m]")
ax.set_ylabel("count")
ax.legend()
plt.tight_layout()
fig.savefig(RESULTS / "object_distance.png", dpi=150)
plt.show()

## 4. Object sizes per class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, dim in zip(axes, ["dim_length", "dim_width", "dim_height"]):
    sns.boxplot(data=eval_labels, x="type", y=dim, ax=ax)
    ax.set_title(dim.replace("dim_", "").capitalize() + " [m]")
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()
fig.savefig(RESULTS / "object_dimensions.png", dpi=150)
plt.show()

## 5. Occlusion & truncation

In [ ]:
occ_map = {0: "fully visible", 1: "partly occluded",
           2: "largely occluded", 3: "unknown"}
eval_labels["occlusion"] = eval_labels["occluded"].map(occ_map)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
eval_labels["occlusion"].value_counts().reindex(
    list(occ_map.values())).plot(kind="bar", ax=axes[0], color="indianred")
axes[0].set_title("Occlusion level")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

axes[1].hist(eval_labels["truncated"], bins=20, color="seagreen")
axes[1].set_title("Truncation (0 = fully inside image)")
axes[1].set_xlabel("truncation")
plt.tight_layout()
fig.savefig(RESULTS / "occlusion_truncation.png", dpi=150)
plt.show()

## 6. Points per scan (sampled)

In [ ]:
# Loading every .bin is slow, so sample a subset for the distribution.
velo_files = sorted((TRAIN / "velodyne").glob("*.bin"))
rng = np.random.default_rng(0)
sample = rng.choice(len(velo_files), size=500, replace=False)

point_counts = [load_point_cloud(velo_files[i]).shape[0] for i in sample]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(point_counts, bins=40, color="slateblue")
ax.set_title("Points per scan (500 random training frames)")
ax.set_xlabel("number of points")
ax.set_ylabel("frames")
plt.tight_layout()
fig.savefig(RESULTS / "points_per_scan.png", dpi=150)
plt.show()

print(f"Mean points per scan: {np.mean(point_counts):,.0f}")

## 7. Bird's-eye view of one scan with ground-truth boxes

Labels live in the **camera** coordinate frame, so to draw boxes on the
LiDAR point cloud we transform their corners back into the **velodyne**
frame using the calibration (`R0_rect` and `Tr_velo_to_cam`).

In [ ]:
from matplotlib.patches import Polygon

def cam_to_velo(points_cam, calib):
    """Transform (N, 3) points from rectified camera coords to velodyne coords."""
    R0 = np.eye(4); R0[:3, :3] = calib["R0_rect"]
    Tr = np.eye(4); Tr[:3, :4] = calib["Tr_velo_to_cam"]
    cam_to_velo_mat = np.linalg.inv(R0 @ Tr)
    pts_h = np.hstack([points_cam, np.ones((len(points_cam), 1))])
    return (cam_to_velo_mat @ pts_h.T).T[:, :3]

def box_footprint_cam(row):
    """Return the 4 base corners of a 3D box in rectified camera coords."""
    l, w = row.dim_length, row.dim_width
    x_c = np.array([ l/2,  l/2, -l/2, -l/2])
    z_c = np.array([ w/2, -w/2, -w/2,  w/2])
    cos, sin = np.cos(row.rotation_y), np.sin(row.rotation_y)
    x_rot =  cos * x_c + sin * z_c
    z_rot = -sin * x_c + cos * z_c
    return np.stack([x_rot + row.loc_x,
                     np.full(4, row.loc_y),
                     z_rot + row.loc_z], axis=1)

frame = "000010"
pc = load_point_cloud(TRAIN / "velodyne" / f"{frame}.bin")
calib = load_calib(TRAIN / "calib" / f"{frame}.txt")
frame_labels = load_labels(TRAIN / "label_2" / f"{frame}.txt")

COLORS = {"Car": "tab:blue", "Pedestrian": "tab:red", "Cyclist": "tab:green"}

fig, ax = plt.subplots(figsize=(9, 9))
m = (np.abs(pc[:, 0]) < 50) & (np.abs(pc[:, 1]) < 40)
ax.scatter(pc[m, 0], pc[m, 1], s=0.3, c=pc[m, 2], cmap="viridis")

for _, row in frame_labels.iterrows():
    if row["type"] not in COLORS:
        continue
    corners = cam_to_velo(box_footprint_cam(row), calib)
    ax.add_patch(Polygon(corners[:, :2], closed=True, fill=False,
                         edgecolor=COLORS[row["type"]], linewidth=2))

ax.set_aspect("equal")
ax.set_xlabel("x - forward [m]")
ax.set_ylabel("y - left [m]")
ax.set_title(f"Frame {frame}: bird's-eye view with ground-truth boxes")
plt.tight_layout()
fig.savefig(RESULTS / f"bev_{frame}.png", dpi=150)
plt.show()

## Summary

- **Class distribution** - heavy imbalance (Car dominates; Cyclist/Pedestrian rare).
- **Distance / size / occlusion** - most objects are nearby and unoccluded;
  small, distant, occluded objects are the hard cases.
- **Points per scan** - ~100k points each; input is sparse and irregular.
- **BEV with boxes** - shows what the model has to learn to localise.

All figures are saved in `results/`.